In [ ]:
import os, sys
cur_directory = os.getcwd()
project_folder = os.path.abspath(os.path.join(cur_directory, os.pardir, os.pardir))
save_directory = os.path.join(cur_directory, 'saved')
sys.path.append(os.path.join(project_folder, 'analysis-suite'))

In [ ]:
import numpy as np
from numpy import linalg as la
import matplotlib as mpl
from matplotlib import pyplot as plt
import analyses as als
import datasets as ds
import models as mds
import tools
from random import sample
from tqdm import tqdm
from argparse import Namespace
from time import gmtime, strftime
import pickle

In [ ]:
data_path = os.path.join(project_folder, 'data', 'nat_scenes_data')
D = ds.DataNatScenes(data_path, mov_ind=4)
print(f'current stimulus is {D.movname}')
mov_ind = D.mov_ind

In [ ]:
V_width = 1
V_E_buffer = 0
E_width = 1
setup = (V_width,V_E_buffer,E_width)
N_trials = 10
N_neurons = 93

dns = ds.DataNatScenes
def emptyfunc(_):
    pass
trial_keys = ["control", "time", "cell", "both"]
shuffle_methods = [emptyfunc, dns.shuffle_trial_per_time, dns.shuffle_trial_per_cell, dns.shuffle_trial]

opt_params = {"track_quantities": False, "quiet": True, "iterations": 16}
mi_measurement_params = {"quiet": True, "batch_size": 500, "N_batches": 500}

critic_params = {
    "hidden_dim": 32,
    "embed_dim": 30,
    "layers": 2,
    "activation": "tanh",
}

experiment_configuration_variables = {'N_neurons', 'trial_keys', 'mi_measurement_params', 'N_trials', 'opt_params', 'V_width', 'V_E_buffer', 'E_width', 'critic_params'}

## Run test

In [ ]:
experiment_dims = (N_trials,)
mi_averages_test = {}
mi_stdevs_test = {}
mi_averages_train = {}
mi_stdevs_train = {}
for key in trial_keys:
    mi_averages_test[key] = np.zeros(experiment_dims)
    mi_stdevs_test[key] = np.zeros(experiment_dims)
    mi_averages_train[key] = np.zeros(experiment_dims)
    mi_stdevs_train[key] = np.zeros(experiment_dims)
    
for n in tqdm(range(N_trials)):
    for k, key in enumerate(trial_keys):   
        D.reset()
        D.trim(20)
        D.subset_features(range(N_neurons))
        _ = shuffle_methods[k](D)
        V, E = D.create_V_and_E_datasets(*setup)
        train_points, test_points = tools.generate_train_test_indices(V.P, 0.5)
        
        critic_function = mds.SeparableCritic(**critic_params)
        
        # training set
        V.subset_points(train_points)
        E.subset_points(train_points)
        als.train_NCE(
            X=V.X,
            Y=E.X,
            critic_function=critic_function,
            **opt_params
        )

        # measure InfoNCE for train
        mi_train = als.MI_NCE(
            X=V.X, Y=E.X, critic_function=critic_function, **mi_measurement_params
        )

        # measure InfoNCE for test
        V.reset()
        E.reset()
        V.subset_points(test_points)
        E.subset_points(test_points)
        mi_test = als.MI_NCE(
            X=V.X, Y=E.X, critic_function=critic_function, **mi_measurement_params
        )

        mi_averages_test[key][n] = mi_test["mi"]
        mi_stdevs_test[key][n] = mi_test["mi_stdev"]
        mi_averages_train[key][n] = mi_train["mi"]
        mi_stdevs_train[key][n] = mi_train["mi_stdev"]
    

## Save

In [ ]:
# select variables
critic_type = type(critic_function)
critic_config = critic_function.get_config()
save_vars = {
    "mi_averages_train",
    "mi_averages_test",
    "mi_stdevs_train",
    "mi_stdevs_test",
    "critic_type",
    "critic_config"
}.union(experiment_configuration_variables)
local_dict = locals()
output_data = {k: local_dict[k] for k in save_vars}

In [ ]:
# save things
timestamp = strftime("%Y.%m.%d %Hh %Mm %Ss.dat",gmtime())
# full_file_path = os.path.join(save_directory, '', strftime("%Y.%m.%d %Hh %Mm %Ss",gmtime())+'.dat')

stim = D.movname
save_directory = os.path.join(cur_directory, 'saved', stim)
# full_file_path = os.path.join(save_directory, f'trial shuffle comparisons {setup} N={N_neurons} high-iter.dat')
# Paper data used more than one iteration; the suffix keeps quick runs from overwriting it.
high_iter_suffix = ' high-iter' if opt_params["iterations"] > 1 else ''
full_file_path = os.path.join(save_directory, f'trial shuffle comparisons {setup} N={N_neurons}{high_iter_suffix}.dat')
# full_file_path = os.path.join(save_directory, 'test.dat')
# print(f"will save to {save_directory}")
print(f"will save to {full_file_path}")

In [ ]:
os.makedirs(os.path.dirname(full_file_path), exist_ok=True)
with open(full_file_path, 'wb+') as f:
            pickle.dump(output_data, f)
tools.record_parameters(full_file_path, output_data)

## Reload and plot

In [ ]:
stim = 'opticflow'
setup = (1,11,1)
N_neurons = 40
V_width, V_E_buffer, E_width = setup

# load_path = f'./saved/{stim}/trial shuffle comparisons {setup} N={N_neurons} high-iter.dat'
load_path = f'./saved/{stim}/trial shuffle comparisons {setup} N={N_neurons}.dat'
with open(load_path, 'rb') as f:
    data_dict = pickle.load(f)
data = Namespace(**data_dict)


In [ ]:
def process_trials(averages, stdevs, trial_axis=-1):
    trial_average = np.average(averages, axis=trial_axis)
    trial_variance = tools.cov_entrywise(averages, sum_axes=trial_axis)
    full_stdevs = np.sqrt(np.sum(stdevs**2, axis=trial_axis) + trial_variance)
    return trial_average, full_stdevs

In [ ]:
average_test = {}
stdevs_test = {}
average_train = {}
stdevs_train = {}

for k, key in enumerate(data.trial_keys):
    average_test[key], stdevs_test[key] = process_trials(data.mi_averages_test[key], data.mi_stdevs_test[key])
    average_train[key], stdevs_train[key] = process_trials(data.mi_averages_train[key], data.mi_stdevs_train[key])

In [ ]:
c1 = 0.3
c2 = 0.8
alp = 0.2
fig, ax = plt.subplots(dpi=300, figsize=[5,5 * 3 / 4])

show_train = False

N_keys = len(data.trial_keys)
# x = np.arange(4)
x = np.arange(N_keys)
w = 0.1 if show_train else 0

for k, key in enumerate(data.trial_keys):
    if show_train:
        y = average_train[key] / np.log(2)
        yerr = stdevs_train[key] / np.log(2)
        ax.errorbar(x[k] - w, y, yerr, marker='o', color=(c1,c2,c1))

    y = average_test[key] / np.log(2)
    yerr = stdevs_test[key] / np.log(2)
    ax.errorbar(x[k] + w, y, yerr, marker='o', color=(c1,c1,c2))
    ax.axhline(y, color=(c2,c2,c2,alp))


ax.set_xticks(x)
ax.set_xticklabels(data.trial_keys)
ax.set_ylabel("Info NCE (bits)")
ax.set_title(f"Trial shuffling with $N={N_neurons}$ neurons,\n $(V,B,E)={setup}$, under {stim} stim")

In [ ]:
dI = {}
for k in data.trial_keys:
    dI[k] = average_test["control"] - average_test[k]

In [ ]:
dI["time"] + dI["cell"] - dI["both"]

In [ ]:
C = N_neurons**2 / D.N_trials

In [ ]:
C